In [1]:
import os
import json
import random
import re
from datetime import datetime
from pathlib import Path

import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# --- Configuration ---
UPDATE_FREQUENCY = 1
M = 7  # LLM pool size

# --- API / Env Setup ---
dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for p in candidates:
        if p.exists():
            dotenv_path = str(p)
            break
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print("Loaded .env:", dotenv_path)
else:
    print("No .env found; please create one at project root.")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")
assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY 未设置，请检查 .env"

Decompose_MODEL_NAME = "deepseek/deepseek-v3.2"
GRADER_MODEL_NAME = "deepseek/deepseek-v3.2"

base_url = OPENROUTER_BASE_URL
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=base_url)

# --- Load model pool from model_config.py ---
import sys
project_root = Path.cwd() if (Path.cwd() / "model_config.py").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


Loaded .env: /home/guisy/MyExperiment/LLM_DAG_Routing_02/.env


In [2]:

from model_config import MODELS_CONFIG

AVAILABLE_LLMS = list(MODELS_CONFIG.keys())
print(f"Loaded {len(AVAILABLE_LLMS)} models from model_config.py")
print(AVAILABLE_LLMS[:7])


Loaded 7 models from model_config.py
['qwen/qwen-2.5-7b-instruct', 'meta-llama/llama-3.1-8b-instruct', 'mistralai/mistral-small-3.2-24b-instruct', 'google/gemma-3-27b-it', 'meta-llama/llama-3.3-70b-instruct', 'qwen/qwen3.5-flash-02-23', 'mistralai/mistral-nemo']


In [3]:


def load_dataset(file_path="../data/final_evaluation_dataset_500.json"):
    candidates = [
        Path(file_path),
        Path.cwd() / "final_evaluation_dataset_500.json",
        Path.cwd().parent / "final_evaluation_dataset_500.json",
        Path.cwd() / "data" / "final_evaluation_dataset_500.json",
        Path.cwd().parent / "data" / "final_evaluation_dataset_500.json",
    ]
    real_path = next((p for p in candidates if p.exists()), None)
    if real_path is None:
        print(f"⚠️ 警告: 数据集文件不存在。已尝试: {[str(p) for p in candidates]}")
        return []
    with open(real_path, "r", encoding="utf-8") as f:
        print(f"✅ 成功加载数据集: {real_path}")
        return json.load(f)


def _to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


class ResultRecorder:
    def __init__(self, output_file="execution_records_random.json"):
        candidates = [Path.cwd() / "record", Path.cwd().parent / "record"]
        record_dir = next((p for p in candidates if p.exists() and p.is_dir()), None)
        if record_dir is None:
            raise FileNotFoundError("未找到已有的 record 目录，请先创建 record 文件夹。")

        self.output_file = str(record_dir / Path(output_file).name)
        self.records = {}

        with open(self.output_file, "w", encoding="utf-8") as f:
            json.dump(self.records, f, ensure_ascii=False, indent=2)

    def add_record(self, problem_id: str, question: str, expected_answers: str, final_status: str, attempts: list):
        self.records[problem_id] = {
            "question": question,
            "answers": expected_answers,
            "final_status": final_status,
            "steps_taken": len(attempts),
            "attempts": _to_jsonable(attempts),
        }
        self.save()

    def save(self):
        with open(self.output_file, "w", encoding="utf-8") as f:
            json.dump(_to_jsonable(self.records), f, ensure_ascii=False, indent=2)


class DAGNode:
    def __init__(self, node_id, task_description):
        self.node_id = node_id
        self.task_description = task_description
        self.predecessors = []
        self.successors = []
        self.layer = 0

        self.input_context = ""
        self.output_content = ""
        self.answer_content = ""
        self.selected_model = None
        self.reward = 0.0

    def add_successor(self, node):
        if node not in self.successors:
            self.successors.append(node)
        if self not in node.predecessors:
            node.predecessors.append(self)


class DAGGraph:
    def __init__(self):
        self.nodes = {}

    def add_node(self, node_id, description):
        if node_id not in self.nodes:
            self.nodes[node_id] = DAGNode(node_id, description)
        return self.nodes[node_id]

    def add_edge(self, source_id, target_id):
        if source_id in self.nodes and target_id in self.nodes:
            self.nodes[source_id].add_successor(self.nodes[target_id])

    def compute_layers(self):
        for node in self.nodes.values():
            node.layer = 0
        for _ in range(len(self.nodes) + 1):
            for node in self.nodes.values():
                for pred in node.predecessors:
                    if node.layer < pred.layer + 1:
                        node.layer = pred.layer + 1
        layers = {}
        for node in self.nodes.values():
            if node.layer not in layers:
                layers[node.layer] = []
            layers[node.layer].append(node)
        return layers



In [4]:

def decompose_query_to_dag(query_text: str, problem_id: str, client) -> DAGGraph:
    system_prompt = """
    You are an expert in logical task decomposition. Please decompose the user's complex problem into a Directed Acyclic Graph (DAG) of derivation steps.

    CRITICAL RULES:
    1. You are a planner, NOT a solver: Do NOT directly compute answers, simplify equations, or solve the problem in the task descriptions.
    2. Context preservation: Ensure each sub-task description explicitly includes all information it needs from the original problem statement.
    3. Dependency sufficiency: When assigning predecessors for a node, ensure those predecessors can provide ALL required information, physical quantities, and mathematical variables for the current sub-task.
    4. Multiple-choice questions: If the original problem has options, include full options in the final selection node.
    5. Execution output rule: In each node description, require the solver to wrap the final concise answer with <answer>...</answer>.
    6. If the Original Question includes explicit multiple-choice options (e.g., A, B, C, D), create a final node to match the extracted answer to the correct option letter. If the Original Question is open-ended and does NOT contain options, DO NOT hallucinate options.
       The final node MUST instruct the model to output the exact entity, number, or option required by the Original Question.

    Output MUST be valid JSON only, with this structure (This is only an example):
    {
      "nodes": [
        {"id": "1", "description": "...Put final concise result in <answer>...</answer>.", "predecessors": []},
        {"id": "2", "description": "...Put final concise result in <answer>...</answer>.", "predecessors": ["1"]}
      ]
    }
    """

    def _normalize_task_desc(desc: str) -> str:
        return (desc or "").strip()

    def _fallback_graph(err_msg: str) -> DAGGraph:
        print(f"❌ Decomposition failed: {err_msg}")
        g = DAGGraph()
        g.problem_description = query_text
        g.add_node(
            "fallback_node",
            _normalize_task_desc(
                "Answer the following question directly and wrap final concise answer in <answer>...</answer>: " + query_text
            ),
        )
        return g

    user_prompt = f"Problem ID: {problem_id}\nOriginal Question: {query_text}"

    try:
        resp = client.chat.completions.create(
            model=Decompose_MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            max_tokens=1400,
            temperature=0.2,
        )
        raw_output = (resp.choices[0].message.content or "").strip()

        json_str = raw_output
        if "```json" in json_str:
            json_str = json_str.split("```json", 1)[1].split("```", 1)[0].strip()
        elif "```" in json_str:
            json_str = json_str.split("```", 1)[1].split("```", 1)[0].strip()

        dag_dict = json.loads(json_str)
    except Exception as e:
        return _fallback_graph(f"model call / JSON parse error: {type(e).__name__}: {e}")

    try:
        graph = DAGGraph()
        graph.problem_description = dag_dict.get("problem_description", query_text)

        nodes_data = dag_dict.get("nodes", [])
        if not isinstance(nodes_data, list) or len(nodes_data) == 0:
            raise ValueError("'nodes' missing or empty")

        schema_new = all(
            isinstance(n, dict) and ("id" in n) and ("description" in n) and ("predecessors" in n)
            for n in nodes_data
        )
        schema_a = all(isinstance(n, dict) and ("node_id" in n) and ("sub_task_description" in n) for n in nodes_data)
        schema_b = all(isinstance(n, dict) and ("id" in n) and ("description" in n) for n in nodes_data)

        if schema_new:
            for n_data in nodes_data:
                node_id = str(n_data["id"])
                task_desc = _normalize_task_desc(str(n_data["description"]))
                graph.add_node(node_id, task_desc)
            for n_data in nodes_data:
                target_id = str(n_data["id"])
                preds = n_data.get("predecessors", [])
                if not isinstance(preds, list):
                    preds = []
                for source_id in preds:
                    graph.add_edge(str(source_id), target_id)

        elif schema_a:
            for n_data in nodes_data:
                node_id = str(n_data["node_id"])
                task_desc = _normalize_task_desc(str(n_data["sub_task_description"]))
                graph.add_node(node_id, task_desc)
            for n_data in nodes_data:
                target_id = str(n_data["node_id"])
                for source_id in n_data.get("dependency_node_id", []):
                    graph.add_edge(str(source_id), target_id)

        elif schema_b:
            for n_data in nodes_data:
                node_id = str(n_data["id"])
                task_desc = _normalize_task_desc(str(n_data["description"]))
                graph.add_node(node_id, task_desc)

            edges_data = dag_dict.get("edges", [])
            if not isinstance(edges_data, list):
                edges_data = []
            for e in edges_data:
                if isinstance(e, (list, tuple)) and len(e) == 2:
                    source_id, target_id = e
                    graph.add_edge(str(source_id), str(target_id))
        else:
            raise ValueError("unsupported node schema")

        if len(graph.nodes) == 0:
            raise ValueError("no valid nodes after parsing")

        print(f"✅ Successfully decomposed query into a DAG with {len(graph.nodes)} nodes.")
        return graph
    except Exception as e:
        return _fallback_graph(f"graph build error: {type(e).__name__}: {e}")


def _extract_answer_tag(text: str) -> str:
    if text is None:
        return ""
    s = str(text)
    matches = re.findall(r"<answer>\s*(.*?)\s*</answer>", s, flags=re.IGNORECASE | re.DOTALL)
    if matches:
        return matches[-1].strip()
    return s.strip()


def _normalize_answer_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\u4e00-\u9fff]", "", s)
    return s


def score_final_answer_by_gt(client, ground_truth_answer: str, final_output: str) -> int:
    judge_prompt = (
        "You are a strict grader. Determine whether the [Model Final Answer] is semantically equivalent "
        "to the [Ground Truth Answer]. Output 1 if correct, 0 if incorrect. "
        "ignore any formatting differences. "
        "Only output a single character: 0 or 1. No explanation.\n\n"
        f"[Ground Truth Answer]\n{ground_truth_answer}\n\n"
        f"[Model Final Answer]\n{final_output}\n"
    )

    try:
        resp = client.chat.completions.create(
            model=GRADER_MODEL_NAME,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=5,
            temperature=0.0,
        )
        score_str = (resp.choices[0].message.content or "").strip()
        m = re.search(r"\b([01])\b", score_str)
        if m:
            return int(m.group(1))
        m2 = re.search(r"([0-9]*\.?[0-9]+)", score_str)
        if m2:
            return 1 if float(m2.group(1)) >= 0.5 else 0
    except Exception as e:
        print(f"⚠️ Final grading failed; fallback rule enabled: {type(e).__name__}: {e}")

    gt = _normalize_answer_text(ground_truth_answer)
    pred = _normalize_answer_text(final_output)
    if not gt or not pred:
        return 0
    if gt == pred or gt in pred or pred in gt:
        return 1
    return 0


def score_node_by_judge_llm(client, node_input_text: str, node_output: str) -> float:
    judge_prompt = (
        "You are a strict grader. Based on the [Task Input Text], evaluate how well the [Node Output] "
        "completes the task. Output only a decimal number between 0 and 1 (up to 2 decimals). No explanation.\n\n"
        f"[Task Input Text]\n{node_input_text}\n\n"
        f"[Node Output]\n{node_output}\n"
    )
    try:
        resp = client.chat.completions.create(
            model=GRADER_MODEL_NAME,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=10,
            temperature=0.0,
        )
        score_str = (resp.choices[0].message.content or "").strip()
        match = re.search(r"([0-9]*\.?[0-9]+)", score_str)
        score = float(match.group(1)) if match else 0.0
    except Exception as e:
        print(f"⚠️ Node grading failed, default to 0: {type(e).__name__}: {e}")
        score = 0.0
    return float(max(0.0, min(1.0, score)))


def build_node_input_text(node: DAGNode) -> str:
    predecessor_qa = []
    for p in node.predecessors:
        concise_ans = p.answer_content if getattr(p, "answer_content", "") else _extract_answer_tag(p.output_content)
        predecessor_qa.append(
            f"Predecessor Node {p.node_id}\n"
            f"Question: {p.task_description}\n"
            f"Answer: {concise_ans}"
        )
    predecessor_qa_text = "\n\n".join(predecessor_qa) if predecessor_qa else "No predecessor nodes."

    input_text = (
        f"Current Node Task: {node.task_description}\n\n"
        f"Predecessor Q&A:\n{predecessor_qa_text}\n\n"
        "You may include reasoning, but you MUST provide the final concise answer wrapped in <answer>...</answer>. "
        "Only the content inside <answer> will be used by downstream nodes and grading."
    )
    return input_text



In [5]:

class RandomRouter:
    """随机路由：不使用神经网络与UCB。"""
    def __init__(self, model_names, seed=42):
        self.model_names = list(model_names)
        self.rng = random.Random(seed)
        self.t = 0

    def select_model(self, _feature_vector=None):
        if not self.model_names:
            raise ValueError("model_names 为空，无法路由")
        chosen = self.rng.choice(self.model_names)
        p = 1.0 / len(self.model_names)
        scores_info = {
            name: {"pred_score": 0.0, "bonus": 0.0, "total_ucb": round(p, 6)}
            for name in self.model_names
        }
        return chosen, scores_info

    def add_observations_and_update(self, observations):
        # 随机策略无参数更新，仅计数
        self.t += len(observations)

    def save_model_state(self, file_path):
        out_dir = os.path.dirname(file_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        state = {
            "router": "random",
            "model_names": self.model_names,
            "t": self.t,
            "rng_state": self.rng.getstate(),
        }
        import pickle
        with open(file_path, "wb") as f:
            pickle.dump(state, f)
        print(f"✅ Random router state saved to: {file_path}")

    def load_model_state(self, file_path):
        import pickle
        with open(file_path, "rb") as f:
            state = pickle.load(f)
        self.t = int(state.get("t", 0))
        rs = state.get("rng_state", None)
        if rs is not None:
            self.rng.setstate(rs)
        print(f"✅ Random router state loaded from: {file_path}")
        return True


def execute_and_evaluate_dag(graph: DAGGraph, problem_desc: str, ground_truth_answer: str, router, client):
    layers_dict = graph.compute_layers()
    sorted_layers = sorted(layers_dict.keys())
    execution_attempts = []

    for layer in sorted_layers:
        for node in layers_dict[layer]:
            model_input_text = build_node_input_text(node)
            node.input_context = model_input_text

            chosen_model, scores_info = router.select_model(None)
            node.selected_model = chosen_model

            llm_input_messages = [{"role": "user", "content": model_input_text}]
            try:
                resp = client.chat.completions.create(
                    model=chosen_model,
                    messages=llm_input_messages,
                    max_tokens=768,
                    temperature=0.1,
                )
                node.output_content = (resp.choices[0].message.content or "").strip()
            except Exception as e:
                print(f"⚠️ Node {node.node_id} model call failed: {type(e).__name__}: {e}")
                node.output_content = ""

            node.answer_content = _extract_answer_tag(node.output_content)

            execution_attempts.append({
                "step": len(execution_attempts) + 1,
                "node_id": node.node_id,
                "predecessor_node_ids": [p.node_id for p in node.predecessors],
                "task_desc": node.task_description,
                "chosen_model": node.selected_model,
                "ucb_scores": scores_info,
                "output": node.output_content,
                "parsed_answer": node.answer_content,
                "llm_input_messages": llm_input_messages,
                "vector_source_text": model_input_text,
                "reward": None,
            })

    terminal_nodes = [n for n in graph.nodes.values() if not n.successors]
    final_output = "\n".join([n.answer_content for n in terminal_nodes]).strip()
    final_correct = score_final_answer_by_gt(client, ground_truth_answer, final_output)

    observations = []
    if final_correct == 1:
        for node in graph.nodes.values():
            node.reward = 1.0
            observations.append((None, node.selected_model, node.reward))
        for a in execution_attempts:
            a["reward"] = 1.0
        final_status = "success"
    else:
        for node in graph.nodes.values():
            node.reward = score_node_by_judge_llm(client, node.input_context, node.answer_content)
            observations.append((None, node.selected_model, node.reward))
        reward_map = {n.node_id: n.reward for n in graph.nodes.values()}
        for a in execution_attempts:
            a["reward"] = float(reward_map.get(a["node_id"], 0.0))
        final_status = "failure"

    return observations, final_output, final_status, execution_attempts, int(final_correct)


def main_training_loop(dataset: list, router, client, recorder):
    for idx, record in enumerate(tqdm(dataset, desc="Processing Queries")):
        raw_query = record.get("problem", record.get("question", record.get("query", "")))
        problem_id = record.get("problem_id", record.get("id", str(idx)))
        gt_answer = record.get("answer", record.get("answers", ""))

        if not raw_query:
            continue

        graph = decompose_query_to_dag(raw_query, str(problem_id), client)
        problem_desc = getattr(graph, "problem_description", raw_query)

        obs, final_out, final_status, attempts, final_correct = execute_and_evaluate_dag(
            graph=graph,
            problem_desc=problem_desc,
            ground_truth_answer=gt_answer,
            router=router,
            client=client,
        )

        recorder.add_record(
            problem_id=str(problem_id),
            question=raw_query,
            expected_answers=gt_answer,
            final_status=final_status,
            attempts=attempts,
        )

        router.add_observations_and_update(obs)

        print(f"\n[problem_id={problem_id}] final_correct={final_correct}, final_status={final_status}")
        print(f"final_output: {final_out[:200]}{'...' if len(final_out) > 200 else ''}")

    print("🎉 所有查询处理完毕（随机路由，无训练更新）！")


def _get_record_dir() -> Path:
    candidates = [Path.cwd() / "record", Path.cwd().parent / "record"]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    p = Path.cwd() / "record"
    p.mkdir(parents=True, exist_ok=True)
    return p


def _resolve_record_path(name_or_path: str, record_dir: Path) -> str:
    p = Path(name_or_path)
    if p.is_absolute():
        p.parent.mkdir(parents=True, exist_ok=True)
        return str(p)
    return str(record_dir / p.name)


def run_with_random_router(
    dataset_path: str = "../data/final_evaluation_dataset_500.json",
    run_size: int | None = None,
    use_previous_state: bool = False,
    state_file: str = "random_router_state.pkl",
    raw_record_file: str = "execution_records_random.json",
    per_question_report_file: str = "per_question_metrics_random.json",
):
    record_dir = _get_record_dir()
    state_path = _resolve_record_path(state_file, record_dir)
    raw_record_path_name = Path(raw_record_file).name
    report_path = _resolve_record_path(per_question_report_file, record_dir)

    dataset = load_dataset(dataset_path)
    if not isinstance(dataset, list) or len(dataset) == 0:
        raise ValueError("数据集为空或格式不正确，无法执行。")

    if run_size is not None:
        run_size = max(0, int(run_size))
        dataset = dataset[:run_size]

    if len(dataset) == 0:
        raise ValueError("run_size 截断后数据为空，请调整 run_size。")

    router = RandomRouter(model_names=AVAILABLE_LLMS, seed=42)
    if use_previous_state and Path(state_path).exists():
        try:
            router.load_model_state(state_path)
        except Exception as e:
            print(f"⚠️ 随机路由状态加载失败，使用新状态: {type(e).__name__}: {e}")

    recorder = ResultRecorder(raw_record_path_name)

    per_question_metrics = []
    for idx, record in enumerate(tqdm(dataset, desc="Random Routing")):
        raw_query = record.get("problem", record.get("question", record.get("query", "")))
        problem_id = str(record.get("problem_id", record.get("id", idx)))
        gt_answer = record.get("answer", record.get("answers", ""))
        if not raw_query:
            continue

        graph = decompose_query_to_dag(raw_query, problem_id, client)
        problem_desc = getattr(graph, "problem_description", raw_query)

        obs, final_out, final_status, attempts, final_correct = execute_and_evaluate_dag(
            graph=graph,
            problem_desc=problem_desc,
            ground_truth_answer=gt_answer,
            router=router,
            client=client,
        )

        recorder.add_record(
            problem_id=problem_id,
            question=raw_query,
            expected_answers=gt_answer,
            final_status=final_status,
            attempts=attempts,
        )

        router.add_observations_and_update(obs)

        terminal_node_ids = [n.node_id for n in graph.nodes.values() if not n.successors]
        attempt_map = {str(a.get("node_id")): a for a in attempts}
        answer_models = [attempt_map[nid].get("chosen_model") for nid in terminal_node_ids if nid in attempt_map]
        used_models = sorted({a.get("chosen_model") for a in attempts if a.get("chosen_model")})

        per_question_metrics.append({
            "problem_id": problem_id,
            "is_correct": int(final_correct),
            "final_status": final_status,
            "answer_model": answer_models[0] if len(answer_models) == 1 else answer_models,
            "used_models": used_models,
            "steps_taken": len(attempts),
            "expected_answer": gt_answer,
            "final_output": final_out,
        })

    router.save_model_state(state_path)

    total = len(per_question_metrics)
    correct = sum(x["is_correct"] for x in per_question_metrics)
    accuracy = (correct / total) if total else 0.0

    report_payload = {
        "config": {
            "dataset_path": dataset_path,
            "run_size": run_size if run_size is not None else "all",
            "use_previous_state": bool(use_previous_state),
            "state_file": state_path,
            "raw_record_file": str(_get_record_dir() / raw_record_path_name),
            "generated_at": datetime.now().isoformat(timespec="seconds"),
            "router": "random",
        },
        "summary": {
            "total_questions": total,
            "correct_questions": correct,
            "accuracy": round(accuracy, 4),
        },
        "per_question": per_question_metrics,
    }

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report_payload, f, ensure_ascii=False, indent=2)

    print("\n✅ 随机路由运行完成")
    print(f"- 准确率: {correct}/{total} = {accuracy:.2%}")
    print(f"- 状态文件: {state_path}")
    print(f"- 逐题报告: {report_path}")
    print(f"- 详细执行记录: {str(_get_record_dir() / raw_record_path_name)}")

    return report_payload


In [6]:


# ===== 使用示例（按需修改） =====
report = run_with_random_router(
    dataset_path="../data/final_evaluation_dataset_500.json",
    run_size=200,
    use_previous_state=False,
    state_file="random_router_state.pkl",
    raw_record_file="execution_records_random_200.json",
    per_question_report_file="per_question_metrics_random_200.json",
)
report["summary"]

✅ 成功加载数据集: ../data/final_evaluation_dataset_500.json


Random Routing:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Successfully decomposed query into a DAG with 1 nodes.


Random Routing:   0%|          | 1/200 [00:16<53:25, 16.11s/it]

✅ Successfully decomposed query into a DAG with 7 nodes.


Random Routing:   1%|          | 2/200 [16:14<31:23:04, 570.63s/it]

✅ Successfully decomposed query into a DAG with 6 nodes.


Random Routing:   1%|          | 2/200 [17:21<28:37:49, 520.56s/it]


KeyboardInterrupt: 